# March Madness 2026 - Simulator

**Quick start:**
1. Make sure `data/trained_params.json` exists (run `train.ipynb` first).
2. Load trained params and feature scaler.
3. Load the 2026 men's Kaggle season stats (no KenPom login required).
4. Run the remaining cells to simulate the bracket.

All model logic lives in `marchmadness/`. This notebook is a thin runner around the package.

**No KenPom required** — all stats come from Kaggle `MRegularSeasonDetailedResults.csv`.

In [ ]:
# Cell 1 - Imports
import os
import sys

sys.path.insert(0, ".")

from marchmadness import (
    load_scaler,
    normalize_team_features,
    load_kaggle_season_stats,
    build_kaggle_schedule_features,
    build_team_features,
    ModelParams,
    load_params,
    build_full_bracket,
    simulate_full_bracket,
    simulate_tournament_round_by_round,
    print_first_round_matchups,
    print_champion_counts,
    print_full_tournament,
)

In [ ]:
# Cell 2 - Load parameters

params = load_params("data/trained_params.json")
print("Loaded trained params from data/trained_params.json")

print(f"  margin_scale={params.margin_scale:.2f}  shock_df={params.shock_df:.2f}  shock_scale={params.shock_scale:.4f}")
print(f"  luck_weight={params.luck_weight:.4f}  recent_form_weight={params.recent_form_weight:.4f}")
print(f"  seed_prior_weight={params.seed_prior_weight:.4f}")
print(f"  w_efg={params.w_efg:.4f}  w_to={params.w_to:.4f}  w_orb={params.w_orb:.4f}  w_ftr={params.w_ftr:.4f}")

N_SIM_FULL = 10_000
N_SIM_ROUND = 1_000
SEED = 142

# Load feature scaler (produced by train.ipynb)
_scaler_path = "data/feature_scaler.json"
if os.path.exists(_scaler_path):
    feature_scaler = load_scaler(_scaler_path)
    print(f"Loaded feature scaler from {_scaler_path}")
else:
    feature_scaler = None
    print(f"No scaler found at {_scaler_path} — raw feature values will be used.")

In [ ]:
# Cell 3 - Load 2026 men's season stats from Kaggle
#
# No browser/login required. Reads MRegularSeasonDetailedResults.csv.

SEASON_YEAR = 2026

kenpom_df = load_kaggle_season_stats([SEASON_YEAR], "data/kaggle", gender="M")[SEASON_YEAR]
print(f"Loaded {len(kenpom_df)} men's teams for {SEASON_YEAR}")

In [ ]:
# Cell 4 — 2026 NCAA Tournament bracket (hardcoded from official bracket)
#
# All First Four results:
#   Howard 86 def. UMBC 83           → Howard is 16-seed in Midwest
#   Texas 68 def. NC State 66        → Texas is 11-seed in West
#   Prairie View A&M def. Lehigh     → Prairie View A&M is 16-seed in South
#   Miami OH assumed to beat SMU     → Miami OH is 11-seed in Midwest
#
# No pending First Four games — all 64 slots are filled.

main_bracket_teams = {
    # EAST
    "Duke":              (1,  "East"),
    "Siena":             (16, "East"),
    "Ohio St.":          (8,  "East"),
    "TCU":               (9,  "East"),
    "St. John's":        (5,  "East"),
    "Northern Iowa":     (12, "East"),
    "Kansas":            (4,  "East"),
    "Cal Baptist":       (13, "East"),
    "Louisville":        (6,  "East"),
    "South Florida":     (11, "East"),
    "Michigan St.":      (3,  "East"),
    "North Dakota St.":  (14, "East"),
    "UCLA":              (7,  "East"),
    "UCF":               (10, "East"),
    "Connecticut":       (2,  "East"),
    "Furman":            (15, "East"),
    # SOUTH
    "Florida":           (1,  "South"),
    "Prairie View A&M":  (16, "South"),   # First Four winner (beat Lehigh)
    "Clemson":           (8,  "South"),
    "Iowa":              (9,  "South"),
    "Vanderbilt":        (5,  "South"),
    "McNeese":           (12, "South"),
    "Nebraska":          (4,  "South"),
    "Troy":              (13, "South"),
    "North Carolina":    (6,  "South"),
    "VCU":               (11, "South"),
    "Illinois":          (3,  "South"),
    "Penn":              (14, "South"),
    "Saint Mary's":      (7,  "South"),
    "Texas A&M":         (10, "South"),
    "Houston":           (2,  "South"),
    "Idaho":             (15, "South"),
    # WEST
    "Arizona":           (1,  "West"),
    "LIU":               (16, "West"),
    "Villanova":         (8,  "West"),
    "Utah St.":          (9,  "West"),
    "Wisconsin":         (5,  "West"),
    "High Point":        (12, "West"),
    "Arkansas":          (4,  "West"),
    "Hawaii":            (13, "West"),
    "BYU":               (6,  "West"),
    "Texas":             (11, "West"),   # First Four winner (beat NC State 68-66)
    "Gonzaga":           (3,  "West"),
    "Kennesaw St.":      (14, "West"),
    "Miami FL":          (7,  "West"),
    "Missouri":          (10, "West"),
    "Purdue":            (2,  "West"),
    "Queens":            (15, "West"),
    # MIDWEST
    "Michigan":          (1,  "Midwest"),
    "Howard":            (16, "Midwest"),  # First Four winner (beat UMBC 86-83)
    "Georgia":           (8,  "Midwest"),
    "Saint Louis":       (9,  "Midwest"),
    "Texas Tech":        (5,  "Midwest"),
    "Akron":             (12, "Midwest"),
    "Alabama":           (4,  "Midwest"),
    "Hofstra":           (13, "Midwest"),
    "Tennessee":         (6,  "Midwest"),
    "Miami OH":          (11, "Midwest"),  # First Four winner (assumed beat SMU)
    "Virginia":          (3,  "Midwest"),
    "Wright St.":        (14, "Midwest"),
    "Kentucky":          (7,  "Midwest"),
    "Santa Clara":       (10, "Midwest"),
    "Iowa St.":          (2,  "Midwest"),
    "Tennessee St.":     (15, "Midwest"),
}

first_four_teams = {}  # All First Four games resolved

print(f"Main bracket: {len(main_bracket_teams)} teams (should be 64)")

In [ ]:
# Cell 5 — Build schedule features for 2026 bracket teams
#
# Derives recent_form and conf_tourney_wins from Kaggle regular-season results.
# No KenPom login required.

sf_all, ct_all = build_kaggle_schedule_features([SEASON_YEAR], "data/kaggle", gender="M")
schedule_features_map = sf_all.get(SEASON_YEAR, {})
conf_tourney_results = ct_all.get(SEASON_YEAR, {})

print(f"Built schedule features for {len(schedule_features_map)} teams.")

In [ ]:
# Cell 5b - Load 2026 Kaggle features  [OPTIONAL]
#
# Attaches four-factor stats, neutral court win%, and coach experience to team features.
# Requires the same Kaggle files as train.ipynb + MTeamSpellings.csv.

import difflib

def _build_kaggle_team_features_for_sim(
    kenpom_names: list,
    kaggle_dir: str = "data/kaggle",
    season: int = 2026,
) -> dict:
    """
    Returns {kenpom_name: {col: val}} by matching KenPom team names to Kaggle
    TeamIDs via MTeamSpellings. Pre-flattens the MultiIndex DataFrame to a plain
    dict to avoid pandas .loc ambiguity on unsorted indexes.
    """
    from pathlib import Path
    import pandas as pd
    from marchmadness import build_kaggle_features

    spellings_path = Path(kaggle_dir) / "MTeamSpellings.csv"
    if not spellings_path.exists():
        print(f"MTeamSpellings.csv not found in {kaggle_dir} — skipping Kaggle features.")
        return {}

    spellings_df = pd.read_csv(spellings_path)
    spelling_to_id = {
        str(row["TeamNameSpelling"]).strip().lower(): int(row["TeamID"])
        for _, row in spellings_df.iterrows()
    }
    all_spellings = list(spelling_to_id.keys())

    try:
        kf_df = build_kaggle_features(season=season, kaggle_dir=kaggle_dir)
    except FileNotFoundError as e:
        print(f"Kaggle feature files not found — skipping. ({e})")
        return {}

    kaggle_lookup: dict = {}
    for (s, tid), row in kf_df.iterrows():
        kaggle_lookup[(int(s), int(tid))] = row.to_dict()

    result = {}
    for kenpom_name in kenpom_names:
        name_lower = kenpom_name.strip().lower()
        team_id = spelling_to_id.get(name_lower)
        if team_id is None:
            matches = difflib.get_close_matches(name_lower, all_spellings, n=1, cutoff=0.75)
            if matches:
                team_id = spelling_to_id[matches[0]]
        if team_id is None:
            continue
        kdict = kaggle_lookup.get((season, team_id))
        if kdict is not None:
            result[kenpom_name] = kdict

    matched = len(result)
    total = len(kenpom_names)
    print(f"Kaggle features matched {matched}/{total} teams for {season}.")
    return result


all_kenpom_names = list(main_bracket_teams.keys()) + list(first_four_teams.keys())
kaggle_team_features_2026 = _build_kaggle_team_features_for_sim(
    all_kenpom_names,
    kaggle_dir="data/kaggle",
    season=SEASON_YEAR,
)

if not kaggle_team_features_2026:
    print("No Kaggle features loaded — using defaults.")
else:
    sample_team = next(iter(kaggle_team_features_2026))
    kf = kaggle_team_features_2026[sample_team]
    print(f"\nSample — {sample_team}:")
    nwp = kf.get("neutral_win_pct")
    ctw = kf.get("coach_tourney_wins")
    efg = kf.get("efg_o")
    print(f"  neutral_win_pct    = {nwp:.3f}" if nwp is not None else "  neutral_win_pct    = N/A")
    print(f"  coach_tourney_wins = {ctw}" if ctw is not None else "  coach_tourney_wins = N/A")
    print(f"  efg_o              = {efg:.3f}" if efg is not None else "  efg_o              = N/A")

In [ ]:
# Cell 6 - Build features + bracket

all_seeds = {**main_bracket_teams, **first_four_teams}

features = build_team_features(
    kenpom_df,
    seeds=all_seeds,
    schedule_features_map=schedule_features_map,
    conf_tourney_results=conf_tourney_results or None,
    kaggle_team_features=kaggle_team_features_2026 or None,
)
print(f"Built features for {len(features)} teams.")

# Apply feature normalization (uses scaler fitted during training)
if feature_scaler is not None:
    features = normalize_team_features(features, feature_scaler)
    print("Applied feature normalization.")


first_four_games, round_of_64 = build_full_bracket(main_bracket_teams, first_four_teams)
print(f"First Four: {len(first_four_games)} games | Round of 64: {len(round_of_64)} games")

print_first_round_matchups(round_of_64)

In [ ]:
# Cell 7 - Monte Carlo: championship probabilities

champion_counts = simulate_full_bracket(
    first_four_games, round_of_64,
    features, params,
    n_sim=N_SIM_FULL,
    seed=SEED,
)

print_champion_counts(champion_counts, n_sim=N_SIM_FULL, top_n=12)

In [ ]:
# Cell 7b - Champion concentration diagnostics

import numpy as np

print(f"Temperature (tau) in loaded params: {params.temperature:.4f}")
print()

total = sum(champion_counts.values())
probs_champ = {t: c / total for t, c in champion_counts.items()}
nonzero = {t: p for t, p in probs_champ.items() if p > 0}

entropy = -sum(p * np.log2(p) for p in nonzero.values())
max_entropy = np.log2(len(main_bracket_teams))
concentration = 1.0 - entropy / max_entropy

top1 = max(probs_champ.values())
top3 = sum(sorted(probs_champ.values(), reverse=True)[:3])

teams_above_1pct  = sum(1 for p in probs_champ.values() if p >= 0.01)
teams_above_5pct  = sum(1 for p in probs_champ.values() if p >= 0.05)
teams_above_10pct = sum(1 for p in probs_champ.values() if p >= 0.10)

print("Champion distribution summary:")
print(f"  entropy            = {entropy:.2f} bits  (max = {max_entropy:.2f})")
print(f"  concentration      = {concentration:.3f}  (0=uniform, 1=all one team)")
print(f"  top-1 probability  = {top1:.1%}")
print(f"  top-3 combined     = {top3:.1%}")
print(f"  teams \u2265 1% chance  = {teams_above_1pct}")
print(f"  teams \u2265 5% chance  = {teams_above_5pct}")
print(f"  teams \u2265 10% chance = {teams_above_10pct}")

In [ ]:
# Cell 8 - Round-by-round: most likely bracket path

round_results, round_details = simulate_tournament_round_by_round(
    first_four_games, round_of_64,
    features, params,
    n_sim=N_SIM_ROUND,
    seed=SEED,
)

print_full_tournament(round_results, round_details, n_sim=N_SIM_ROUND)

In [ ]:
# Cell 8b - Round-by-round high-confidence game counts
#
# Count how many games in each round hit >=90% or >=95% win probability.

ROUND_ORDER = ["Round of 64", "Round of 32", "Sweet 16", "Elite 8", "Final Four", "Championship"]

print(f"Temperature (tau) = {params.temperature:.4f}")
print()
print(f"{'Round':<18}  {'Games':>5}  {'\u226590%':>5}  {'\u226595%':>5}  {'\u226590% share':>10}")
print("-" * 52)

total_games = 0
total_hi90  = 0
total_hi95  = 0

for rnd_name in ROUND_ORDER:
    details = round_details.get(rnd_name, [])
    if not details or not isinstance(details, list):
        continue
    n = len(details)
    hi90 = sum(1 for d in details if d[4] >= 0.90)
    hi95 = sum(1 for d in details if d[4] >= 0.95)
    share = hi90 / n if n else 0
    total_games += n
    total_hi90  += hi90
    total_hi95  += hi95
    print(f"{rnd_name:<18}  {n:5d}  {hi90:5d}  {hi95:5d}  {share:10.1%}")

print("-" * 52)
if total_games:
    print(f"{'Total':<18}  {total_games:5d}  {total_hi90:5d}  {total_hi95:5d}  {total_hi90/total_games:.1%}")